# Step 4 — Data Cleaning
### Credit Card Underwriting Pipeline

---

## What is Data Cleaning?

Data cleaning = finding and fixing quality problems that would mislead the model.

> **Analogy:** Imagine weighing yourself with a broken scale that sometimes reads -50kg.
> Any analysis using that data would be wrong. Data cleaning finds and fixes the broken scale.

---

## Problems We Address in This Notebook

| Problem | Example | Fix |
|---------|---------|-----|
| **Outliers** | Age = 200 (impossible) | Cap at business-valid maximum |
| **Wrong data types** | Income stored as string | Convert to float |
| **Impossible values** | Negative FICO score | Clip to valid range [300, 850] |
| **Encoding categoricals** | "Full-Time" as text | Encode to integer |
| **Constant columns** | All values = 0 | Drop (zero variance = zero information) |
| **Duplicate rows** | Same applicant submitted twice | De-duplicate |

---

## Why Clean Before Modelling?

- Outliers distort the model's sense of what is "normal" — it may try to fit that one extreme value
- A categorical column with 50 unique values fed without encoding will crash most models
- Constant columns add noise and memory usage with zero benefit
- Duplicate rows inflate some applicants' weight, biasing the model


In [ ]:
# ── Re-run Setup + Steps 1-3 ──────────────────────────────────────────────────
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')

DATA_PATH   = 'cc_underwriting_5k_stratified11.csv'
IGNORE_COLS = ['applicant_id', 'target_approved', 'target_credit_limit_assigned']
HIGH_MISS_THRESH = 50.0
SEED = 42

df_raw       = pd.read_csv(DATA_PATH)
numeric_cols = [c for c in df_raw.select_dtypes(include='number').columns if c not in IGNORE_COLS]
cat_cols     = [c for c in df_raw.select_dtypes(include=['object','string']).columns if c not in IGNORE_COLS]

# Step 3: remove high-miss columns
def missing_report(df):
    m = df.isnull().sum(); p = m/len(df)*100
    r = pd.DataFrame({'missing_count':m,'missing_pct':p.round(2)})
    return r[r['missing_count']>0].sort_values('missing_pct',ascending=False)

miss = missing_report(df_raw)
drop_cols = miss[miss['missing_pct'] > HIGH_MISS_THRESH].index.tolist()
df = df_raw.drop(columns=drop_cols)
numeric_cols = [c for c in numeric_cols if c in df.columns]
cat_cols     = [c for c in cat_cols     if c in df.columns]

for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

df['target'] = (df['target_approved'] == 'Yes').astype(int)
print(f'Data ready: {df.shape}  |  nulls: {df.isnull().sum().sum()}')

## 4.1 — Check for Duplicate Rows

> **Duplicate rows bias the model** because a duplicated applicant gets double the weight.
> In credit modelling this is especially risky — one bad loan application could be counted twice,
> making the model think that profile is twice as common as it really is.


In [ ]:
# Check for exact duplicate rows (all columns identical)
n_duplicates = df.duplicated().sum()
print(f'Duplicate rows: {n_duplicates}')

if n_duplicates > 0:
    # Drop duplicates — keep the first occurrence
    df = df.drop_duplicates()
    print(f'Removed {n_duplicates} duplicates. New shape: {df.shape}')
else:
    print('No duplicates found. Dataset is clean on this dimension.')

# Check for duplicated applicant IDs only (same person, different row)
id_dups = df_raw['applicant_id'].duplicated().sum()
print(f'Duplicate applicant_ids: {id_dups}')

## 4.2 — Outlier Detection with IQR Method

> **Outlier:** a value so extreme it is unlikely to be real, or so far from the rest that it
> distorts model training.
>
> **IQR (Interquartile Range) method:**
> - Q1 = 25th percentile, Q3 = 75th percentile
> - IQR = Q3 - Q1
> - Lower fence = Q1 - 1.5 * IQR
> - Upper fence = Q3 + 1.5 * IQR
> - Any value outside the fences = outlier
>
> **Why 1.5?** This is Tukey's rule from 1977. For a normal distribution, it captures ~0.7% of values.
> You can use 3.0 for a more conservative definition (only the most extreme outliers).


In [ ]:
# Detect outliers using the IQR method
KEY_CHECK = ['age', 'annual_income', 'fico_score', 'debt_to_income_ratio',
             'credit_utilization_ratio', 'net_worth', 'requested_credit_limit']

print(f'{"Column":<35} {"Q1":>10} {"Q3":>10} {"IQR":>10} {"Lower":>10} {"Upper":>10} {"Outliers":>10}')
print('-' * 100)

outlier_info = {}
for col in KEY_CHECK:
    if col not in df.columns:
        continue
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR

    n_outliers = ((df[col] < lower_fence) | (df[col] > upper_fence)).sum()
    outlier_info[col] = {'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
                          'lower': lower_fence, 'upper': upper_fence,
                          'n_outliers': n_outliers}

    print(f'{col:<35} {Q1:>10.2f} {Q3:>10.2f} {IQR:>10.2f} '
          f'{lower_fence:>10.2f} {upper_fence:>10.2f} {n_outliers:>10}')

## 4.3 — Outlier Treatment: Capping (Winsorisation)

> **Two main strategies for outliers:**
>
> 1. **Remove the row** — only if you are certain the value is an error (e.g., age = 200)
> 2. **Cap (Winsorise)** — replace extreme values with the fence value
>
> **We use capping** because:
> - In credit data, very high incomes are real — we do not want to drop those applicants
> - Capping preserves the row while limiting the outlier's influence on the model
> - This is the standard approach in credit scorecard development
>
> **Business-aware caps:** We use domain-valid caps, not just statistical fences,
> because `fico_score` is always 300–850 regardless of what the IQR says.


In [ ]:
# Business-valid min/max caps for key credit features
# These come from domain knowledge, not just statistics
BUSINESS_CAPS = {
    'age'                    : (18, 100),     # legal borrowing age to realistic max
    'fico_score'             : (300, 850),    # FICO score is always 300-850
    'debt_to_income_ratio'   : (0.0, 2.0),   # 200% DTI is extreme but possible
    'credit_utilization_ratio': (0.0, 1.0),  # cannot be above 100%
    'annual_income'          : (0, 5_000_000),# cap at $5M — above this is data error
}

cap_log = []
for col, (lo, hi) in BUSINESS_CAPS.items():
    if col not in df.columns:
        continue

    n_below = (df[col] < lo).sum()
    n_above = (df[col] > hi).sum()

    if n_below > 0 or n_above > 0:
        # np.clip replaces values below lo with lo, above hi with hi
        df[col] = df[col].clip(lower=lo, upper=hi)
        cap_log.append({'column': col, 'lo': lo, 'hi': hi,
                         'n_clipped_low': n_below, 'n_clipped_high': n_above})
        print(f'  {col}: clipped {n_below} values below {lo}, {n_above} above {hi}')

if not cap_log:
    print('No values outside business caps. Data is clean.')

## 4.4 — Visualise Outlier Treatment (Before vs After)

> **Box plots make outliers visible.** The whiskers extend to the IQR fences;
> dots beyond the whiskers are flagged as outliers. After capping, you should see
> no dots (or fewer) outside the fences.


In [ ]:
# Before/after comparison for a key feature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

col = 'debt_to_income_ratio'

# Before: use raw data
axes[0].boxplot(df_raw[col].dropna(), patch_artist=True,
                 boxprops={'facecolor': '#e74c3c', 'alpha': 0.7})
axes[0].set_title(f'{col} — Before Cleaning', fontweight='bold')
axes[0].set_ylabel('Value')

# After: use cleaned data
axes[1].boxplot(df[col].dropna(), patch_artist=True,
                 boxprops={'facecolor': '#2ecc71', 'alpha': 0.7})
axes[1].set_title(f'{col} — After Capping', fontweight='bold')
axes[1].set_ylabel('Value')

plt.suptitle('Outlier Treatment Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4.5 — Remove Constant and Near-Constant Columns

> **A column with one unique value carries zero information.**
> It cannot help the model distinguish between approved and declined because
> all applicants have the same value — it is literally the same for everyone.
>
> **Near-constant:** > 99% of values are the same — negligible variation, remove to reduce noise.


In [ ]:
# Check for zero-variance (constant) columns
n_unique_per_col = df[numeric_cols].nunique()

# Constant: only 1 unique value
constant_cols = n_unique_per_col[n_unique_per_col <= 1].index.tolist()

# Near-constant: > 99.5% of rows have the same value
near_constant_cols = []
for col in numeric_cols:
    top_freq = df[col].value_counts(normalize=True).iloc[0]
    if top_freq > 0.995:
        near_constant_cols.append((col, top_freq))

print(f'Constant columns (1 unique value): {len(constant_cols)}')
for c in constant_cols:
    print(f'  {c}')

print(f'Near-constant columns (>99.5% same value): {len(near_constant_cols)}')
for c, freq in near_constant_cols:
    print(f'  {c}: {freq:.1%} identical')

# Drop constants
if constant_cols:
    df.drop(columns=constant_cols, inplace=True)
    numeric_cols = [c for c in numeric_cols if c not in constant_cols]
    print(f'Dropped {len(constant_cols)} constant columns')

## 4.6 — Categorical Encoding: Label Encoding

> **Machine learning models only understand numbers, not strings.**
> "Full-Time", "Part-Time", "Self-Employed" must become 0, 1, 2 (or similar).
>
> ### Encoding Options
>
> | Method | How It Works | Use When |
> |--------|-------------|----------|
> | **Label Encoding** | Assigns integers 0, 1, 2... | Tree models (order does not matter for RF) |
> | **One-Hot Encoding** | Creates one binary column per category | Regression/linear models (no ordinal assumption) |
> | **Target Encoding** | Replaces category with its mean target rate | High-cardinality; risk of leakage |
> | **WoE Encoding** | Replaces category with its WoE value | Credit scorecards (covered in Step 5) |
>
> **We use Label Encoding** because Random Forest splits on values without assuming order.
> Tree splits like "is employment_status_enc <= 1?" work correctly regardless of the integer assignment.


In [ ]:
# Label-encode categorical columns with manageable cardinality
# Skip columns with > 30 unique values (too many categories for label encoding to be meaningful)
MAX_CARDINALITY = 30

le = LabelEncoder()
encoded_cols = []
skipped_cols = []

for col in cat_cols:
    n_unique = df[col].nunique()
    if n_unique <= MAX_CARDINALITY:
        # fit_transform: learn the mapping AND apply it in one step
        # We convert to string first to handle any mixed types gracefully
        df[col + '_enc'] = le.fit_transform(df[col].astype(str))
        encoded_cols.append(col + '_enc')
    else:
        skipped_cols.append((col, n_unique))

print(f'Label-encoded {len(encoded_cols)} categorical columns:')
for col in encoded_cols[:10]:
    orig = col.replace('_enc','')
    print(f'  {orig} -> {col}  (unique values: {df[orig].nunique()})')

print()
if skipped_cols:
    print(f'Skipped {len(skipped_cols)} high-cardinality columns:')
    for c, n in skipped_cols:
        print(f'  {c}: {n} unique values')

CAT_ENC_FEATURES = encoded_cols

## 4.7 — Data Type Corrections

> **Pandas infers data types when loading a CSV.** Sometimes it gets it wrong.
> For example: a column of 0s and 1s might be loaded as float when it should be int.
> A date column might be loaded as string (object).
>
> Correct types ensure:
> - Less memory usage
> - Correct operations (you cannot sum a string column)
> - Proper behaviour in model training


In [ ]:
# Check current dtypes for key columns
print('Current data types of key columns:')
print()
key_cols_check = ['age', 'fico_score', 'annual_income', 'debt_to_income_ratio',
                  'target_approved', 'target']

for col in key_cols_check:
    if col in df.columns:
        dtype = df[col].dtype
        sample = df[col].iloc[0]
        print(f'  {col:<35} dtype={str(dtype):<12} sample={sample}')

# Fix: age should be int, not float
if 'age' in df.columns and df['age'].dtype == float:
    df['age'] = df['age'].astype(int)
    print()
    print('Fixed: age converted from float to int')

# Check: binary flag columns should be int (0/1), not float
flag_cols = [c for c in df.columns if 'flag' in c.lower() and c.endswith('_enc')]
for col in flag_cols:
    if df[col].dtype == float:
        df[col] = df[col].astype(int)

print()
print(f'Final DataFrame shape: {df.shape}')
print(f'Remaining nulls: {df.isnull().sum().sum()}')

## Step 4 Complete — Data Cleaning Summary

**What we did:**

| Action | Why |
|--------|-----|
| Checked for duplicates | Prevent inflated weights for repeated applicants |
| IQR outlier detection | Identify extreme values objectively |
| Business cap (Winsorisation) | Clip impossible values using domain knowledge |
| Removed constant columns | Zero-variance features carry zero information |
| Label-encoded categoricals | Convert strings to integers for model training |
| Verified data types | Ensure correct operations and memory efficiency |

**Outcome:** `df` is clean, type-correct, and has all categorical features encoded as integers.

**Next:** `05_Feature_Engineering.ipynb` — IV, WoE, interaction features, log transforms, multicollinearity
